In [1]:
# Langchain Embedding API

# 라이브러리 불러오기
import numpy as np
from numpy import dot
from numpy.linalg import norm
import pandas as pd
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings( 
    model="text-embedding-qwen3-embedding-8b",
    base_url="http://...:.../v1",
    api_key="lm-studio",
    check_embedding_ctx_length=False,
)

query_result = embeddings.embed_query('저는 배가 고파요')
print(query_result)

[0.017648400738835335, 0.009287641383707523, -0.0023873329628258944, -0.04438294097781181, 0.014782784506678581, -0.008044143207371235, -0.013311907649040222, -0.0056120045483112335, -0.00044960473314858973, 0.015197430737316608, -0.015585342422127724, -0.002215705346316099, 0.07027337700128555, -0.02565133199095726, 0.023162251338362694, 0.02400273084640503, 0.008959474042057991, -0.011576864868402481, 0.04650207236409187, 0.021422000601887703, 0.007223639637231827, 0.025404851883649826, 0.024856124073266983, -0.000676970521453768, 0.016197189688682556, 0.02231478877365589, -0.035327665507793427, 0.006801317445933819, -0.020232882350683212, 0.0413970947265625, -0.010402610525488853, -0.02367299795150757, -0.00781321246176958, 0.018650604411959648, -0.032230935990810394, 0.0017340087797492743, -0.027715714648365974, -0.009205734357237816, -0.004504940006881952, -0.008187657222151756, -0.01522859837859869, 0.010195701383054256, -0.0014211746165528893, 0.011135452426970005, -0.0207322034

In [2]:
data = [
    '주식 시장이 급등했어요',
    '시장 물가가 올랐어요',
    '전통 시장에는 다양한 물품들을 팔아요',
    '부동산 시장이 점점 더 복잡해지고 있어요',
    '저는 빠른 비트를 좋아해요',
    '최근 비트코인 가격이 많이 변동했어요'
]

df = pd.DataFrame(data, columns=['text'])

df

,text
0,주식 시장이 급등했어요
1,시장 물가가 올랐어요
2,전통 시장에는 다양한 물품들을 팔아요
3,부동산 시장이 점점 더 복잡해지고 있어요
4,저는 빠른 비트를 좋아해요
5,최근 비트코인 가격이 많이 변동했어요


In [3]:
# 텍스트 임베딩 벡터로 변환하는 함수 정의
def get_embedding(text):
    return embeddings.embed_query(text)

# DataFrame의 각 행에 대해 'text' 열의 내용을 임베딩 벡터로 변환
df['embedding'] = df.apply(
    lambda row: get_embedding(row.text),
    axis=1
)

# 변환된 DataFrame 출력
df

,text,embedding
0,주식 시장이 급등했어요,"[0.003425918985158205, 0.012388941831886768, -..."
1,시장 물가가 올랐어요,"[0.0124553507193923, 0.023535393178462982, -0...."
2,전통 시장에는 다양한 물품들을 팔아요,"[-0.01394876092672348, 0.023344269022345543, -..."
3,부동산 시장이 점점 더 복잡해지고 있어요,"[0.009744472801685333, 0.014037452638149261, -..."
4,저는 빠른 비트를 좋아해요,"[0.031136970967054367, 0.002553999423980713, 0..."
5,최근 비트코인 가격이 많이 변동했어요,"[0.024296490475535393, 0.022296005859971046, 0..."


In [4]:
# 코사인 유사도 계산 함수
def cos_sim(A, B):
    return dot(A, B) / (norm(A) * norm(B))

# 주어진 쿼리와 가장 유사한 상위 3개의 문서를 반환하는 함수
def return_answer_candidate(df, query):
    # 쿼리 텍스트를 임베딩 벡터로 변환
    query_embedding = get_embedding(query)

    # DataFrame의 각 문서 임베딩과 쿼리 임베딩 간의 유사도 계산
    df["similarity"] = df.embedding.apply(lambda x: cos_sim(np.array(x), np.array(query_embedding)))

    # 유사도가 높은 순으로 정렬하고 상위 3개 문서 선택
    top_three_doc = df.sort_values("similarity", ascending=False).head(3)

    return top_three_doc

# 예시 쿼리로 유사한 문서 검색
sim_result = return_answer_candidate(df, '과일 값이 비싸다')
sim_result

,text,embedding,similarity
1,시장 물가가 올랐어요,"[0.0124553507193923, 0.023535393178462982, -0....",0.700582
5,최근 비트코인 가격이 많이 변동했어요,"[0.024296490475535393, 0.022296005859971046, 0...",0.584192
0,주식 시장이 급등했어요,"[0.003425918985158205, 0.012388941831886768, -...",0.580358


In [6]:
# 다른 모델로 테스트
embeddings = OpenAIEmbeddings( 
    model="text-embedding-nomic-embed-text-v1.5",
    base_url="http://...:.../v1",
    api_key="lm-studio",
    check_embedding_ctx_length=False,
)

# DataFrame의 각 행에 대해 'text' 열의 내용을 임베딩 벡터로 변환
df['embedding'] = df.apply(
    lambda row: get_embedding(row.text),
    axis=1
)

# 예시 쿼리로 유사한 문서 검색
sim_result = return_answer_candidate(df, '과일 값이 비싸다')
sim_result

,text,embedding,similarity
5,최근 비트코인 가격이 많이 변동했어요,"[-0.03933055326342583, -0.00351810734719038, -...",0.816629
3,부동산 시장이 점점 더 복잡해지고 있어요,"[-0.0006923280889168382, 0.012346523813903332,...",0.785077
0,주식 시장이 급등했어요,"[-0.00934308860450983, -0.00351078063249588, -...",0.761756
